In [1]:
!pip install langgraph langchain-groQ python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.1 MB/s eta 0:00:00


In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq



In [3]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Groq key (gsk_...): ")

In [4]:
load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b")

In [5]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [6]:
def generate_joke(state: JokeState):
    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content        

    return {'joke': response}

In [7]:
def generate_explanation(state: JokeState):
    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [8]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()                        

workflow = graph.compile(checkpointer=checkpointer)   

In [9]:
config1 = {"configurable": {"thread_id": "1"}}


In [10]:

workflow.invoke({"topic": "pizza"}, config=config1)



{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?  \n\nBecause it wanted to earn some *dough* while staying *on a roll*—and it heard the workplace had great “slice” benefits! 🍕😄',
 'explanation': '**Explanation of the joke**\n\n| Part of the joke | What it literally means | What the wordplay / pun refers to |\n|------------------|------------------------|-----------------------------------|\n| **“Why did the pizza apply for a job?”** | A pizza is an inanimate food item, so it can’t actually look for work. | Sets up a classic “why‑did‑the‑X…?” joke format that invites a punny answer. |\n| **“Because it wanted to earn some *dough* …”** | *Dough* is the raw batter that becomes pizza crust. | “Dough” is also slang for money. The pizza wants to *make money*—just as a person would when they get a job. |\n| **“…while staying *on a roll* …”** | A roll is a small loaf of bread; a pizza isn’t a roll. | “On a roll” is an idiom meaning you’re experiencing a streak of success. The jok

In [ ]:
workflow.get_state(config1)



NameError: name 'workflow' is not defined

In [15]:
tuple(workflow.get_state_history(config1))

(StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?  \n\nBecause it wanted to earn some *dough* while staying *on a roll*—and it heard the workplace had great “slice” benefits! 🍕😄', 'explanation': '**Explanation of the joke**\n\n| Part of the joke | What it literally means | What the wordplay / pun refers to |\n|------------------|------------------------|-----------------------------------|\n| **“Why did the pizza apply for a job?”** | A pizza is an inanimate food item, so it can’t actually look for work. | Sets up a classic “why‑did‑the‑X…?” joke format that invites a punny answer. |\n| **“Because it wanted to earn some *dough* …”** | *Dough* is the raw batter that becomes pizza crust. | “Dough” is also slang for money. The pizza wants to *make money*—just as a person would when they get a job. |\n| **“…while staying *on a roll* …”** | A roll is a small loaf of bread; a pizza isn’t a roll. | “On a roll” is an idiom meaning you’re experiencing a streak